# Author and prove out a least-privilege IAM role and bucket policy for a three-team data lake

The previous platform lead handed the analytics team `AmazonS3FullAccess` and called it a day. Three months later, an ML notebook pulled the compliance team's encrypted financial-records prefix, security flagged it, and the lead was replaced. You inherit the bucket and the three team consumers, and you have one session to build the scoped roles, layer the bucket-policy defense in depth, and prove the boundary holds with a deny-test before handing off credentials.

The three teams use the same bucket but never the same prefix:

- **analytics/** payment-funnel reports the analytics team owns.
- **ml/** training datasets the ML team owns.
- **compliance/** audit-log exports the compliance team owns.

Each team is supposed to be locked to its own prefix. The deny-test at the end is where you prove it.

**Time.** Plan for about 50 minutes hands-on. IAM propagation can push debugging sessions past 60 minutes; budget up to 90 if the trust policy fights back. The cleanup cell at the bottom keeps your AWS costs near zero either way.

**Cost.** This lab costs zero dollars if you follow the steps and clean up. IAM and STS are always free. The S3 footprint of one bucket and three tiny seed objects is well under the always-free tier.

This lab maps to AWS SAA-C03 Domain 1: Design Secure Architectures (30% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.5.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "iam-policies-and-trust"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[0].slug exactly
REGION = "us-east-1"  # SAA-C03 Batch 1 labs run in us-east-1 per brief

TEAMS = ["analytics", "ml", "compliance"]
ROLE_NAMES = {team: f"labrun-{LAB_ID}-{team}-role" for team in TEAMS}
INLINE_POLICY_NAMES = {team: f"labrun-{team}-prefix-policy" for team in TEAMS}
PREFIXES = {team: f"{team}/" for team in TEAMS}

# Bucket name is resolved after STS GetCallerIdentity returns the account ID.
# S3 bucket names must be globally unique; the account-ID suffix keeps the
# notebook deterministic without colliding with other students.
BUCKET_NAME = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA Batch 1 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. This lab has no
# critical (hourly-billed) resources; IAM, STS, and S3 at this scale are
# always-free. Per RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks
# execution if any tagged resources from a prior session exist.
#
# Note on the bucket: deleting the bucket via the s3_bucket adapter clears
# objects, the bucket policy, and tagging together. No separate bucket-policy
# entry is needed. The iam_role adapter clears inline policies before
# deleting each role, so no separate inline-policy entries are needed.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=ROLE_NAMES["compliance"],
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAMES['compliance']}",
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAMES["ml"],
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAMES['ml']}",
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAMES["analytics"],
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAMES['analytics']}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("remove them manually with the AWS CLI commands printed by the")
    print("cleanup cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the data lake bucket and seed each team prefix

S3 has no real folders. The three team prefixes exist only as side effects of object keys that start with `analytics/`, `ml/`, or `compliance/`. The deny-test in Task 4 needs at least one object under each prefix so there is something concrete to either fetch or be denied on.

Build it in this order:

1. Create the bucket (`labrun-iam-policies-and-trust-{your-account-id}`) and tag it with `labrun:lab-slug=iam-policies-and-trust` at creation. The cleanup tag audit needs that tag to find the bucket.
2. Upload three tiny seed objects, one per team. Any one-line content works; the bytes do not matter, only the keys do.

The lab is not about the data. It is about who can read what. The seed objects exist so that `GetObject` in the deny-test has something non-empty to either find or be denied.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the bucket, tag it with the lab tag, seed each team prefix.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket.
# YOUR CODE: Create the S3 bucket using s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects LocationConstraint in create_bucket; other regions require
# CreateBucketConfiguration={"LocationConstraint": REGION}.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Seed one tiny placeholder object per team prefix. The content is arbitrary;
# only the prefix in the Key matters.
for team in TEAMS:
    key = f"{PREFIXES[team]}seed.txt"
    body = f"placeholder seed for {team} team\n".encode("utf-8")
    # YOUR CODE: Upload the placeholder object using s3.put_object() with
    # Bucket=BUCKET_NAME, Key=key, and Body=body.
    print(f"Uploaded s3://{BUCKET_NAME}/{key}")

print()
print(f"All three team prefixes seeded under s3://{BUCKET_NAME}/")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Bucket exists with the lab tag and every team prefix has at
# least one object.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            s3_client.head_bucket(Bucket=BUCKET_NAME)
        except ClientError as e:
            error_code = e.response["Error"]["Code"]
            if error_code in ("404", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Bucket {BUCKET_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        try:
            tag_resp = s3_client.get_bucket_tagging(Bucket=BUCKET_NAME)
            tags = {t["Key"]: t["Value"] for t in tag_resp.get("TagSet", [])}
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchTagSet":
                tags = {}
            else:
                return CheckpointResult(status="error", error_reason=str(e))
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bucket {BUCKET_NAME} is missing tag "
                    f"{LAB_TAG_KEY}={LAB_TAG_VALUE}. Found tags: {tags}"
                ),
            )

        empty_prefixes = []
        for team in TEAMS:
            resp = s3_client.list_objects_v2(
                Bucket=BUCKET_NAME, Prefix=PREFIXES[team], MaxKeys=1
            )
            if not resp.get("Contents"):
                empty_prefixes.append(PREFIXES[team])
        if empty_prefixes:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"These prefixes have no objects: {', '.join(empty_prefixes)}. "
                    f"Each team prefix needs at least one seed object."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint says either the bucket is missing, the tag is missing, or at least one prefix has no objects. S3 has no real folders. Creating a "directory" is just uploading an object whose key starts with that prefix.

</details>

<details><summary>Hint 2 (direction)</summary>

Two calls. `s3.create_bucket(Bucket=BUCKET_NAME)` (no `CreateBucketConfiguration` in `us-east-1`), then a loop over `TEAMS` calling `s3.put_object(Bucket=BUCKET_NAME, Key=f"{team}/seed.txt", Body=body)`. The tag is already set by the `put_bucket_tagging` call that ships in this cell.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET_NAME)
else:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION},
    )

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

for team in TEAMS:
    key = f"{PREFIXES[team]}seed.txt"
    body = f"placeholder seed for {team} team\n".encode("utf-8")
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=body)
```

</details>

**Wallet check.** Still at $0.00. Bucket creation, three tiny seed objects, and tagging fit inside the always-free S3 tier. Your coffee this morning cost infinitely more.

## Task 2: Create three IAM roles, each scoped to a single team prefix

The compliance audit fails because one role can read everything. The fix is one role per team with two pieces correctly aligned: a trust policy that controls who can assume the role, and an inline policy that controls what the role can do once assumed.

Build it once per team:

1. Create the IAM role with a trust policy that allows the current AWS account to call `sts:AssumeRole` on it. Use the account-root principal `arn:aws:iam::{account-id}:root` so any identity in your account with `sts:AssumeRole` permission can assume the role. This is what the deny-test in Task 4 needs.
2. Attach an inline policy that grants `s3:GetObject` and `s3:PutObject` on the team's prefix path (e.g., `arn:aws:s3:::{bucket}/analytics/*`), plus `s3:ListBucket` on the bucket itself with a `Condition` block that pins `s3:prefix` to the team's prefix.
3. Tag the role with `labrun:lab-slug=iam-policies-and-trust` so the cleanup audit can find it.

Three roles. Three trust policies. Three inline policies. After this task each role can only see and write to its own team prefix. The cell ends with a 15-second IAM propagation wait because `sts.assume_role` is stricter than most other IAM calls.

In [ ]:
# NBVAL_SKIP
# Task 2: Create three IAM roles, each with an inline policy scoped to a
# single team prefix and an s3:prefix Condition on ListBucket.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "sts:AssumeRole",
        }
    ],
}

for team in TEAMS:
    role_name = ROLE_NAMES[team]
    prefix = PREFIXES[team]

    try:
        # YOUR CODE: Create the IAM role with iam.create_role(). Pass
        # RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust_policy),
        # Description=f"labrun {team} team role", and
        # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
        print(f"Created role: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise

    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:PutObject"],
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}/{prefix}*",
            },
            {
                "Effect": "Allow",
                "Action": "s3:ListBucket",
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
                "Condition": {
                    "StringLike": {"s3:prefix": [f"{prefix}*"]}
                },
            },
        ],
    }
    # YOUR CODE: Attach the inline policy using iam.put_role_policy() with
    # RoleName=role_name, PolicyName=INLINE_POLICY_NAMES[team], and
    # PolicyDocument=json.dumps(inline_policy).
    print(f"Attached inline policy {INLINE_POLICY_NAMES[team]} to {role_name}")

# IAM propagation. Newly created roles can take a few seconds to be assumable.
print("Waiting 15 seconds for IAM propagation before the deny-test cell runs...")
time.sleep(15)
print("Done.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: All three roles exist with the account-root trust policy and
# an inline policy scoped to the correct team prefix (Resource ARN plus
# s3:prefix Condition on ListBucket).

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        iam_client = boto3.client(
            "iam",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        expected_principal = f"arn:aws:iam::{ACCOUNT_ID}:root"
        required_actions = {"s3:GetObject", "s3:PutObject", "s3:ListBucket"}
        action_wildcards = {"s3:*", "*"}

        for team in TEAMS:
            role_name = ROLE_NAMES[team]
            prefix = PREFIXES[team]

            try:
                role_resp = iam_client.get_role(RoleName=role_name)
            except ClientError as e:
                if e.response["Error"]["Code"] == "NoSuchEntity":
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"Role {role_name} does not exist.",
                    )
                return CheckpointResult(status="error", error_reason=str(e))

            trust = role_resp["Role"]["AssumeRolePolicyDocument"]
            trust_principals = []
            for stmt in trust.get("Statement", []):
                p = stmt.get("Principal", {})
                aws = p.get("AWS")
                if isinstance(aws, str):
                    trust_principals.append(aws)
                elif isinstance(aws, list):
                    trust_principals.extend(aws)
            if expected_principal not in trust_principals:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} trust policy is missing AWS principal "
                        f"{expected_principal!r}. Found: {trust_principals}"
                    ),
                )

            inline_resp = iam_client.list_role_policies(RoleName=role_name)
            policy_names = inline_resp.get("PolicyNames", [])
            if not policy_names:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Role {role_name} has no inline policy attached.",
                )

            doc_resp = iam_client.get_role_policy(
                RoleName=role_name, PolicyName=policy_names[0]
            )
            policy_doc = doc_resp["PolicyDocument"]
            policy_str = json.dumps(policy_doc)
            expected_resource = f"arn:aws:s3:::{BUCKET_NAME}/{prefix}*"
            expected_prefix_value = f"{prefix}*"
            if expected_resource not in policy_str:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} inline policy does not include the "
                        f"prefix-scoped Resource ARN {expected_resource!r}. "
                        f"Check the s3:GetObject/s3:PutObject Resource."
                    ),
                )
            if expected_prefix_value not in policy_str:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} inline policy is missing an s3:prefix "
                        f"Condition value of {expected_prefix_value!r}. "
                        f"ListBucket must be scoped via Condition on s3:prefix."
                    ),
                )

            all_actions = set()
            for stmt in policy_doc.get("Statement", []):
                actions = stmt.get("Action", [])
                if isinstance(actions, str):
                    all_actions.add(actions)
                else:
                    all_actions.update(actions)
            has_wildcard = bool(all_actions & action_wildcards)
            missing = required_actions - all_actions
            if missing and not has_wildcard:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} inline policy is missing required "
                        f"actions: {sorted(missing)}. Add them explicitly or "
                        f"use the s3:* wildcard."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two pieces have to line up per role: the trust policy (who can assume) and the inline policy (what the assumed role can do). The checkpoint failure message says which one of the two is wrong on which role. Read it before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

Trust policy `Principal` is the account-root ARN: `arn:aws:iam::{ACCOUNT_ID}:root`. Inline policy needs THREE actions: `s3:GetObject`, `s3:PutObject`, and `s3:ListBucket`. `s3:GetObject` and `s3:PutObject` are scoped via the `Resource` ARN (e.g., `arn:aws:s3:::{bucket}/analytics/*`). `s3:ListBucket` is scoped via a `Condition` block on `s3:prefix`, not via Resource. The Condition is the part students forget.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "sts:AssumeRole",
        }
    ],
}

for team in TEAMS:
    role_name = ROLE_NAMES[team]
    prefix = PREFIXES[team]
    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"labrun {team} team role",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
    except ClientError as e:
        if e.response["Error"]["Code"] != "EntityAlreadyExists":
            raise

    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:PutObject"],
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}/{prefix}*",
            },
            {
                "Effect": "Allow",
                "Action": "s3:ListBucket",
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
                "Condition": {
                    "StringLike": {"s3:prefix": [f"{prefix}*"]}
                },
            },
        ],
    }
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName=INLINE_POLICY_NAMES[team],
        PolicyDocument=json.dumps(inline_policy),
    )
```

</details>

**Wallet check.** Still at $0.00. IAM `CreateRole`, `PutRolePolicy`, and a 15-second sleep do not move the AWS bill. The compliance team is the only thing accruing cost right now (their time, waiting on the audit response).

## Task 3: Layer a bucket policy with three cross-team Deny statements

Inline policies set what each role IS allowed to do. The bucket policy sets what every role is NOT allowed to do, regardless of what its inline policy says. This is defense in depth: if a future engineer widens the analytics role's inline policy to cover `arn:aws:s3:::{bucket}/*`, the bucket-policy Deny still keeps it out of `ml/` and `compliance/`.

Three Deny statements, one per role. Each statement names one role as `Principal.AWS` and lists the OTHER two team prefix ARNs as `Resource`. Action covers at least `s3:GetObject` and `s3:PutObject`.

AWS policy evaluation: explicit Deny in any attached policy wins over every Allow. The Deny statements here close the door even if a future inline policy drifts.

In [ ]:
# NBVAL_SKIP
# Task 3: Build the bucket policy with three cross-team Deny statements and
# attach it via put_bucket_policy.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deny_statements = []
for team in TEAMS:
    role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES[team]}"
    other_prefixes = [PREFIXES[other] for other in TEAMS if other != team]
    other_resources = [
        f"arn:aws:s3:::{BUCKET_NAME}/{p}*" for p in other_prefixes
    ]
    deny_statements.append(
        {
            "Sid": f"Deny{team.capitalize()}CrossTeam",
            "Effect": "Deny",
            "Principal": {"AWS": role_arn},
            "Action": ["s3:GetObject", "s3:PutObject"],
            "Resource": other_resources,
        }
    )

bucket_policy = {
    "Version": "2012-10-17",
    "Statement": deny_statements,
}

# YOUR CODE: Attach the bucket policy using s3.put_bucket_policy() with
# Bucket=BUCKET_NAME and Policy=json.dumps(bucket_policy).

print(f"Bucket policy attached with {len(deny_statements)} Deny statements.")
print(json.dumps(bucket_policy, indent=2))

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Bucket policy contains three Deny statements that each name
# one role as Principal and the other two team prefixes as Resource. Accepts
# wildcard actions (s3:*, *) alongside literal s3:GetObject/s3:PutObject.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            policy_resp = s3_client.get_bucket_policy(Bucket=BUCKET_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchBucketPolicy":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Bucket {BUCKET_NAME} has no bucket policy attached.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        policy_doc = json.loads(policy_resp["Policy"])
        statements = policy_doc.get("Statement", [])
        action_wildcards = {"s3:*", "*"}
        required_actions = {"s3:GetObject", "s3:PutObject"}

        for team in TEAMS:
            role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES[team]}"
            other_prefixes = [PREFIXES[other] for other in TEAMS if other != team]
            other_resources = {
                f"arn:aws:s3:::{BUCKET_NAME}/{p}*" for p in other_prefixes
            }

            matching_deny = None
            for stmt in statements:
                if stmt.get("Effect") != "Deny":
                    continue
                p = stmt.get("Principal", {})
                aws_principal = p.get("AWS")
                if isinstance(aws_principal, str):
                    principals = [aws_principal]
                elif isinstance(aws_principal, list):
                    principals = aws_principal
                else:
                    principals = []
                if role_arn not in principals:
                    continue
                actions = stmt.get("Action", [])
                if isinstance(actions, str):
                    actions = [actions]
                action_set = set(actions)
                covers_required = required_actions.issubset(action_set) or bool(
                    action_set & action_wildcards
                )
                if not covers_required:
                    continue
                resources = stmt.get("Resource", [])
                if isinstance(resources, str):
                    resources = [resources]
                if other_resources.issubset(set(resources)):
                    matching_deny = stmt
                    break

            if matching_deny is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Bucket policy is missing a Deny statement that names "
                        f"{role_arn!r} as Principal and denies s3:GetObject and "
                        f"s3:PutObject (or a wildcard covering them) on the other "
                        f"teams' prefixes. Expected Resources: {sorted(other_resources)}."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks the bucket policy looking for a Deny statement per role. If it cannot find one for a specific role, the message tells you which role ARN it expected and which Resource ARNs should appear. Read that first.

</details>

<details><summary>Hint 2 (direction)</summary>

Three statements, one per role. For role `analytics-role`, `Principal.AWS` is the analytics role ARN and `Resource` is a list containing the `ml/` and `compliance/` prefix object ARNs. Actions need both `s3:GetObject` and `s3:PutObject` (or a covering wildcard like `s3:*`). Skipping the put action leaves a write-side hole that the validator catches.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deny_statements = []
for team in TEAMS:
    role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES[team]}"
    other_prefixes = [PREFIXES[other] for other in TEAMS if other != team]
    other_resources = [
        f"arn:aws:s3:::{BUCKET_NAME}/{p}*" for p in other_prefixes
    ]
    deny_statements.append(
        {
            "Sid": f"Deny{team.capitalize()}CrossTeam",
            "Effect": "Deny",
            "Principal": {"AWS": role_arn},
            "Action": ["s3:GetObject", "s3:PutObject"],
            "Resource": other_resources,
        }
    )

bucket_policy = {"Version": "2012-10-17", "Statement": deny_statements}
s3.put_bucket_policy(Bucket=BUCKET_NAME, Policy=json.dumps(bucket_policy))
```

</details>

**Wallet check.** Still at $0.00. Bucket-policy attach is a free S3 control-plane call. The wallet does not move until someone makes a billable data-plane request, and the deny-test in Task 4 only fires two GetObject calls against tiny placeholder objects.

## Task 4: Prove the boundary with a programmatic deny-test

Two things have to be true: the analytics role can read its own prefix, and the analytics role cannot read the ml or compliance prefixes. Anything else is a broken design.

Build it in this order:

1. Call `sts.assume_role` against the analytics role ARN to get temporary credentials.
2. Build a NEW boto3 S3 client from those temp credentials (including `aws_session_token`). Reusing your base credentials tests your own permissions, not the role.
3. Call `s3.get_object` on `ml/seed.txt`. Catch `ClientError` and assert the Error Code is `AccessDenied`.
4. Call `s3.get_object` on `analytics/seed.txt`. The same client should succeed.

The deny-test is the actual proof. Inline policy plus bucket policy plus trust policy all working together is what the cert exam tests.

In [ ]:
# NBVAL_SKIP
# Task 4: Programmatic deny-test. Assume the analytics role, get a fresh S3
# client from the temporary credentials, then attempt cross-team reads and
# same-team reads.

sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES['analytics']}"

# YOUR CODE: Call sts.assume_role(RoleArn=analytics_role_arn,
# RoleSessionName="labrun-deny-test") and store the response in assume_resp.

temp_creds = assume_resp["Credentials"]

# YOUR CODE: Build a NEW boto3 S3 client from temp_creds. Pass
# aws_access_key_id=temp_creds["AccessKeyId"],
# aws_secret_access_key=temp_creds["SecretAccessKey"],
# aws_session_token=temp_creds["SessionToken"], region_name=REGION.
# Assign it to s3_assumed.

# Cross-team reads should fail.
for other in ["ml", "compliance"]:
    key = f"{other}/seed.txt"
    try:
        s3_assumed.get_object(Bucket=BUCKET_NAME, Key=key)
        print(f"UNEXPECTED PASS: read s3://{BUCKET_NAME}/{key} as analytics")
    except ClientError as e:
        code_str = e.response["Error"]["Code"]
        print(f"Denied as expected: {key} returned {code_str}")

# Same-team read should succeed.
resp = s3_assumed.get_object(Bucket=BUCKET_NAME, Key="analytics/seed.txt")
print(f"Same-team read OK: analytics/seed.txt returned {resp['ResponseMetadata']['HTTPStatusCode']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Independent deny-test. Assume the analytics role, build a
# fresh client, and verify GetObject is denied on ml/ and compliance/ but
# allowed on analytics/. The checkpoint does not read student state; it
# re-runs the proof against live AWS.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES['analytics']}"

        # IAM eventual consistency. Wrap assume_role + reads in a retry loop
        # capped at 60 seconds with exponential backoff.
        last_error = None
        delay = 2
        elapsed = 0
        while elapsed < 60:
            try:
                assume_resp = sts_client.assume_role(
                    RoleArn=role_arn,
                    RoleSessionName="labrun-checkpoint-4",
                )
                last_error = None
                break
            except ClientError as e:
                last_error = e
                time.sleep(delay)
                elapsed += delay
                delay = min(delay * 2, 8)
        if last_error is not None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn} within 60s. STS returned "
                    f"{last_error.response['Error']['Code']}: "
                    f"{last_error.response['Error']['Message']}. "
                    f"Check the trust policy Principal."
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )

        for other in ["ml", "compliance"]:
            key = f"{other}/seed.txt"
            try:
                s3_assumed.get_object(Bucket=BUCKET_NAME, Key=key)
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Analytics role successfully read s3://{BUCKET_NAME}/{key}. "
                        f"The bucket policy is not denying cross-team GetObject for {other}/."
                    ),
                )
            except ClientError as e:
                code_str = e.response["Error"]["Code"]
                if code_str != "AccessDenied":
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"Expected AccessDenied on {other}/, got {code_str}. "
                            f"Verify the bucket policy Deny Principal and Resource."
                        ),
                    )

        try:
            s3_assumed.get_object(Bucket=BUCKET_NAME, Key="analytics/seed.txt")
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Analytics role could not read its own prefix analytics/seed.txt. "
                    f"Got {code_str}. Verify the inline policy Resource and that the "
                    f"bucket policy is not over-broad."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes are possible: either the analytics role cannot read its own prefix (something denied a path that should be allowed), or the analytics role CAN read the ml or compliance prefix (the Deny statements are missing or wrong). The checkpoint message says which one.

</details>

<details><summary>Hint 2 (direction)</summary>

The most common miss is forgetting to build a NEW boto3 client from the temporary credentials. Reusing your base `s3` client tests your own permissions, not the role. Confirm `aws_session_token=temp_creds["SessionToken"]` is passed to the new client. If assume_role itself fails, the trust policy `Principal` does not match your caller account, or IAM has not propagated yet (the validator already retries for 60 seconds).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES['analytics']}"

assume_resp = sts.assume_role(
    RoleArn=analytics_role_arn,
    RoleSessionName="labrun-deny-test",
)
temp_creds = assume_resp["Credentials"]

s3_assumed = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)

for other in ["ml", "compliance"]:
    try:
        s3_assumed.get_object(Bucket=BUCKET_NAME, Key=f"{other}/seed.txt")
    except ClientError as e:
        assert e.response["Error"]["Code"] == "AccessDenied"

s3_assumed.get_object(Bucket=BUCKET_NAME, Key="analytics/seed.txt")
```

</details>

**Wallet check.** Still at $0.00. STS AssumeRole is always free. Three GetObject calls against tiny placeholder objects fit comfortably inside the always-free S3 request quota. The deny-test does not move the meter.

## Cleanup

Time to tear it all down. The cell below runs through your manifest in reverse-creation order, deletes every resource, then double-checks AWS to confirm nothing is left. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 01 has no critical (hourly-billed) resources, so the canonical
# summary always reports zero critical destructions.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** IAM, STS, bucket policies, and a handful of S3 requests against tiny placeholder objects all fit under the always-free tier. The bucket and roles were destroyed by the cleanup cell above, so nothing is still accruing. Your morning coffee cost infinitely more than the AWS bill from this session.

## Reflection

These are not graded. They are for you.

1. Walk through what AWS evaluates when a single principal has an Allow on `s3:GetObject` in its IAM inline policy AND a Deny on `s3:GetObject` in the bucket policy for the same object. Which wins, and why does AWS evaluate it in that order?

2. Why does layering a bucket-policy Deny on top of the inline-policy scope buy you anything? The inline policy already does not grant the other team prefixes; what attack or drift scenario does the explicit Deny actually mitigate?

## Exam-Style Practice

**Question 1 (Domain 1, policy evaluation order):**

Your IAM policy attached to an IAM role grants `s3:GetObject` on `arn:aws:s3:::data-lake/analytics/*`. The bucket policy on `data-lake` contains a statement with `Effect: Deny`, `Principal: { AWS: <role ARN> }`, `Action: s3:GetObject`, `Resource: arn:aws:s3:::data-lake/analytics/q4-revenue.csv`. The role calls GetObject on `data-lake/analytics/q4-revenue.csv`. What is the result?

A. The call succeeds because the IAM policy explicitly grants GetObject and IAM policies override bucket policies.

B. The call returns `AccessDenied` because an explicit `Deny` in any attached policy overrides every `Allow`.

C. The call succeeds because the IAM policy was attached more recently than the bucket policy.

D. The call returns `AccessDenied` because IAM roles cannot access bucket-policy-protected objects without an STS session.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: IAM and bucket policies are combined, not preferred. Neither overrides the other by attachment location.
- B is correct: AWS policy evaluation evaluates every Allow and Deny across IAM, resource, SCP, and permissions boundary policies; an explicit Deny anywhere wins.
- C is wrong: policy evaluation does not consider attachment timestamps.
- D is wrong: IAM roles using temporary credentials routinely access bucket-policy-protected objects when both policies allow.

</details>

**Question 2 (Domain 1, ListBucket scoping):**

You attach this inline policy to a role:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    { "Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject"], "Resource": "arn:aws:s3:::lab-bucket/analytics/*" },
    { "Effect": "Allow", "Action": "s3:ListBucket", "Resource": "arn:aws:s3:::lab-bucket" }
  ]
}
```

The role calls `aws s3 ls s3://lab-bucket/`. What does it see?

A. Only the keys under `analytics/`, because the GetObject scope filters the ListBucket response.

B. Every key in the bucket including `ml/` and `compliance/` keys, because ListBucket has no `s3:prefix` Condition.

C. `AccessDenied`, because `s3:ListBucket` requires `s3:ListBucketVersions` to be granted alongside it.

D. Only the three top-level prefixes (`analytics/`, `ml/`, `compliance/`) but not the objects inside them.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: IAM does not filter ListBucket responses based on GetObject scope. ListBucket and GetObject are independent authorization decisions.
- B is correct: without `Condition: StringLike: s3:prefix: analytics/*` on the ListBucket statement, the role can enumerate every key in the bucket.
- C is wrong: `s3:ListBucketVersions` is a separate action for versioned-bucket version listing, not a prerequisite for `s3:ListBucket`.
- D is wrong: S3 has no real folders; ListBucket returns flat keys, not a directory tree.

</details>